<a href="https://colab.research.google.com/github/kauanesv/linkedIn-queens-solver/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bibliotecas**

In [101]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import random

# **Constantes**

In [102]:
TAMANHO_CONJUNTO = 8
QUANTIDADE_IMAGENS = 14
TAMANHO_INDIVIDUO = 8
SEED = 42

In [103]:
random.seed(SEED + 2)

# **GitHub**

In [104]:
# !rm -rf linkedIn-queens-solver
# !git clone https://github.com/kauanesv/linkedIn-queens-solver.git
# %cd linkedIn-queens-solver

# **Carrega Imagens**

In [105]:
# função que carrega as imagens (prints) do tabuleiro
def carregar_imagens_tabuleiro(id_imagem):
  # carrega imagem do github
  print_tabuleiro = cv2.imread(f'imagens_tabuleiro/{id_imagem:03d}.jpeg')
  # exibe print de tela do tabuleiro
  # plt.imshow(print_tabuleiro, cmap='gray')
  # plt.title(f"Print {id_imagem:03d} do Tabuleiro LinkedIn (8x8)")
  # plt.axis('off')
  # plt.show()
  # retorna a imagem carregada
  return print_tabuleiro

print_tabuleiro = carregar_imagens_tabuleiro(1)

# **Recorte Automático do Tabuleiro**

In [106]:
# função que recorta a região de interesse (tabuleiro) do print automaticamente
def recortar_imagem_tabuleiro(print_tabuleiro):
  # recorte inicial empírico
  imagem_tabuleiro = print_tabuleiro[260:1000, :]
  # converte para escala de cinza
  cinza = cv2.cvtColor(imagem_tabuleiro, cv2.COLOR_BGR2GRAY)
  # binariza a imagem recortada
  _, threshold = cv2.threshold(cinza, 30, 255, cv2.THRESH_BINARY)
  # normaliza imagem para padrão UINT8 (0 a 255)
  imagem_binaria = ((~threshold) * 255).astype(np.uint8)
  # exibe imagem binária recortada
  # plt.imshow(~threshold, cmap='gray')
  # plt.title("Imagem binária recortada empiricamente")
  # plt.axis('off')
  # plt.show()
  # encontrar os contornos da imagem binaria
  contornos, _ = cv2.findContours(imagem_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
  # cria buffer da imagem binaria para visualizar contornos
  imagem_binaria_rgb = cv2.cvtColor(imagem_binaria, cv2.COLOR_GRAY2RGB)
  cv2.drawContours(imagem_binaria_rgb, contornos, -1, (255, 0, 0), 2)
  # exibe os contornos encontrados na imagem
  # plt.figure(figsize=(12, 6))
  # plt.imshow(imagem_binaria_rgb)
  # plt.title("Contornos encontrados na imagem binária")
  # plt.axis('off')
  # plt.show()
  # guarda o maior contorno (que será o tabuleiro 8x8)
  maior_contorno = max(contornos, key=cv2.contourArea)
  # guarda informações geométricas do contorno encontrado (dimensões do quadrado/tabuleiro)
  x, y, w, h = cv2.boundingRect(maior_contorno)
  # recortar apenas a região do tabuleiro
  tabuleiro_recortado = imagem_tabuleiro[y:y+h, x:x+w]
  # exibe tabuleiro recortado
  # plt.imshow(tabuleiro_recortado, cmap='gray')
  # plt.title("Imagem do tabuleiro recortado a partir do maior contorno")
  # plt.axis('off')
  # plt.show()
  # retorna o tabuleiro recortado
  return tabuleiro_recortado

tabuleiro_recortado = recortar_imagem_tabuleiro(print_tabuleiro)

TypeError: 'NoneType' object is not subscriptable

# **Matriz do Tabuleiro**

In [107]:
# função que gera a matriz de cores do tabuleiro a partir da imagem do tabuleiro
def gerar_matriz_tabuleiro(tabuleiro_recortado):
  # redimensiona o tabuleiro para um tamanho fixo padrão para facilitar os cálculos
  tabuleiro_padrao = cv2.resize(tabuleiro_recortado, (400, 400))
  # calcula o tamanho aproximado de cada célula (quadradinho no tabuleiro)
  tamanho_celula = 400 // 8
  # inicia lista que amazenará as cores de cada célula
  cores_tabuleiro = []
  # percorre todo o tabuleiro (linhas x colunas)
  for linha in range(TAMANHO_CONJUNTO):
    for coluna in range(TAMANHO_CONJUNTO):
      # calcula o pixel central exato de cada quadradinho para evitar pegar as linhas pretas da borda
      centro_y = (linha * tamanho_celula) + (tamanho_celula // 2)
      centro_x = (coluna * tamanho_celula) + (tamanho_celula // 2)
      # coleta a cor daquele ponto/célula no tabuleiro
      cor_pixel = tabuleiro_padrao[centro_y, centro_x]
      # guarda a cor na lista de cores das células (quadradinho)
      cores_tabuleiro.append(cor_pixel)
  # aplica algoritmo de clusterização sobre a lista de cores das células/quadradinhos
  # K = 8 (clusters) e semente fixa (42)
  kmeans = KMeans(n_clusters=8, random_state=SEED)
  # armezena predição de clusters da lista de cores do tabuleiro
  labels_cores = kmeans.fit_predict(cores_tabuleiro)
  # transformar a lista de 64 resultados em uma matriz 8x8
  tabuleiro_matriz_cores = labels_cores.reshape(8, 8)
  # exibe mapeamento da matriz do tabuleiro
  # print("\nMapeamento dos quadradinhos (matriz):")
  # print(tabuleiro_matriz_cores)
  # print("\n")
  # exibe tabuleiro recortado para comparação com matriz
  plt.imshow(tabuleiro_padrao, cmap='gray')
  plt.title("Imagem do tabuleiro recortado")
  plt.axis('off')
  plt.show()
  # retorna matriz mapeada do tabuleiro
  return tabuleiro_matriz_cores

matriz_tabuleiro = gerar_matriz_tabuleiro(tabuleiro_recortado)

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


# **Cálculo de Conflitos**

In [ ]:
# função que calcula conflitos de rainhas no tabuleiro
def calcular_conflitos(estado, tabuleiro_cores):
  # inicia com 0 conflitos
  conflitos = 0
  # percorre as as posições no tabuleiro
  for i in range(TAMANHO_CONJUNTO):
    for j in range(i + 1, TAMANHO_CONJUNTO):
      # verifica se há rainhas em diagonais adjacentes
      if abs(i - j) == 1 and abs(estado[i] - estado[j]) == 1:
        conflitos += 1
      # guarda a cor (grupo) que pertencem duas rainhas
      cor_rainha1 = tabuleiro_cores[i][estado[i]]
      cor_rainha2 = tabuleiro_cores[j][estado[j]]
      # verifica se essas duas rainhas estão na mesma cor (grupo)
      if cor_rainha1 == cor_rainha2:
        conflitos += 1
      # verifica se há rainhas na mesma linha
      if estado[i] == estado[j]:
        conflitos += 1
  # exibe a quantidade de conflitos no tabuleiro
  # print(f"Quantidade de conflitos: {conflitos}\n")
  # retorna a quantidade de conflitos
  return conflitos

# estado aleatório para teste
estado = [1, 3, 4, 7, 0, 2, 5, 6]
conflitos = calcular_conflitos(estado, matriz_tabuleiro)


# **BFS**

In [108]:

# Algoritmo de Busca em Largura para encontrar a solução do jogo Queens do LinkedIn
def bfs(matriz_tabuleiro):

    import collections

    # regras
    def posicaoValida(estadoAtual, novaColuna):
        novaLinha = len(estadoAtual)
        novaCor = matriz_tabuleiro[novaLinha][novaColuna]

        for linhaExistente, colunaExistente in enumerate(estadoAtual):
            # restrição de coluna
            if colunaExistente == novaColuna:
                return False
            # restrição de cor
            corExistente = matriz_tabuleiro[linhaExistente][colunaExistente]
            if corExistente == novaCor:
                return False
            # restrição de vizinhos diretos
            if abs(linhaExistente - novaLinha) <= 1 and abs(colunaExistente - novaColuna) <= 1:
                return False
        return True

    # função para gerar as próximas jogadas
    def gerarProximasJogadas(tabuleiro_atual):
        # inicializa uma lista vazia para armazenar as próximas jogadas válidas (estados filhos)
        novasJogadas = []
        # se o tabuleiro (pai) já possui 8 rainhas, finalia o jogo
        if len(tabuleiro_atual) >= 8:
            return novasJogadas
        # percorre cada uma das 8 colunas possíveis
        for coluna in range(8):
            # verifica se colocar uma rainha nessa coluna descumpre alguma regra
            if posicaoValida(tabuleiro_atual, coluna):
                # se for uma posição segura, cria a nova configuração a partir da nova coluna
                novasJogadas.append(tabuleiro_atual + [coluna])

        # retorna o conjunto de todos as próximas jogadas válidas encontradas
        return novasJogadas

    # inicia o jogo
    estadoInicial = []
    # inicializa a fila de busca
    abertos = collections.deque([estadoInicial])
    # cria um conjunto de tabuleiros já testados
    fechados = set()
    # cria o dicionário para mapear o histórico de jogadas
    caminho = {str(estadoInicial): None}
    # inicializa o contador de iterações
    iteracoes = 0

    # enquanto existirem tabuleiros na fila para testar
    while len(abertos) > 0:
        # retira o tabuleiro mais antigo da fila.
        estadoAtual = abertos.popleft()
        # se o tabuleiro atual tem 8 rainhas a solução foi encontrada
        if len(estadoAtual) == 8:
            return caminho, estadoAtual, iteracoes
        # não visita os tabuleiros repetidos
        if tuple(estadoAtual) not in fechados:
            # adiciona o tabuleiro atual aos estados já visitados
            fechados.add(tuple(estadoAtual))
            # gera as próximas jogadas válidas (filhos do estado atual)
            novasJogadas = gerarProximasJogadas(estadoAtual)
            # analisa cada filho gerado (nova jogada)
            for jogada in novasJogadas:
                # se é uma jogada nova e não está na fila
                if tuple(jogada) not in fechados and jogada not in abertos:
                    # Coloca a jogada no fim da fila
                    abertos.append(jogada)
                    # salva o caminho que este tabuleiro gerou essa jogada
                    caminho[str(jogada)] = str(estadoAtual)

        # incrementa as iterações
        iteracoes += 1

    # se não houver solução
    return "FALHA", None, iteracoes

# **DFS**

In [109]:
# Algoritmo de Busca em Profundidade
def dfs(matriz_tabuleiro, n_max=8):

    # regras do Queens
    def posicaoValida(estadoAtual, novaColuna):
        novaLinha = len(estadoAtual)
        novaCor = matriz_tabuleiro[novaLinha][novaColuna]

        for linhaExistente, colunaExistente in enumerate(estadoAtual):
            # restrição de coluna
            if colunaExistente == novaColuna:
                return False
            # restrição de cor
            corExistente = matriz_tabuleiro[linhaExistente][colunaExistente]
            if corExistente == novaCor:
                return False
            # restrição de vizinhos diretos
            if abs(linhaExistente - novaLinha) <= 1 and abs(colunaExistente - novaColuna) <= 1:
                return False
        return True

    # gera colunas válidas
    def gerarProximasJogadas(tabuleiro_atual):
        novasJogadas = []
        # a profundidade máxima do tabuleiro é 8 linhas
        if len(tabuleiro_atual) >= 8:
            return novasJogadas
        for coluna in range(8):
            if posicaoValida(tabuleiro_atual, coluna):
                # o tabuleiro atual(pai) gera uma nova jogada(filho)
                novasJogadas.append(tabuleiro_atual + [coluna])
        # retorna a nova jogada
        return novasJogadas

    # reconstroi o caminho
    def reconstruir_caminho(caminho, estadoAtual):

        # lista para a sequência de jogadas
        rota = []
        # começa pelo estado final
        atual = estadoAtual
        # volta o histórico até o início
        while atual is not None:
            # guarda o tabuleiro atual na rota
            rota.append(atual)
            # converte em texto para buscar a chave
            atual_str = str(atual)
            # se o tabuleiro está no histórico
            if atual_str in caminho:
                # puxa o tabuleiro PAI anterior
                atual = caminho[atual_str]
            else:
                # para o loop
                atual = None
        # retorna o caminho completo
        return rota

    # inicia o jogo
    estadoInicial = []
    # inicializa a fila de busca
    abertos = [estadoInicial]
    # cria um conjunto de tabuleiros já testados
    fechados = set()
    # cria o dicionário para mapear o histórico de jogadas
    caminho = {str(estadoInicial): None}
    # inicializa o contador de iterações
    iteracoes = 0

    # enquanto existirem tabuleiros na fila para testar
    while len(abertos) > 0:
        estadoAtual = abertos.pop(0)
        # se o tabuleiro atual tem 8 rainhas a solução foi encontrada
        if len(estadoAtual) == 8:
            return caminho, estadoAtual, iteracoes
        # não visita os tabuleiros repetidos
        if tuple(estadoAtual) not in fechados:
            fechados.add(tuple(estadoAtual))
            # gera os movimentos se respeitando o limite máximo de profundidade(linhas)
            rota = reconstruir_caminho(caminho, estadoAtual)
            if(len(estadoAtual) < n_max):
                # Chama a função de geração de novas jogadas
                filhos = gerarProximasJogadas(estadoAtual)
                for filho in filhos:
                    # se o estado filho não estiver na lista de fechados ou abertos
                    if tuple(filho) not in fechados and filho not in abertos:
                        # adiciona ao topo da pilha de estados abertos
                        abertos.insert(0, filho)
                        # salva o estado pai (tabuleiro) dele no histórico de caminhos
                        caminho[str(filho)] = estadoAtual

            # atualiza quantidade de estados percorridos
            iteracoes += 1

    # se não houver solução
    return "FALHA", None, iteracoes

# **AG**

## **Seleção**

In [110]:
# ---------------------------------------------------------
# Função de Seleção por Torneio
def selecaoPorTorneio(populacao, valoresDeFitness, k = 3):

    #sorteia os individuos
    selecionados = random.sample(range(len(populacao)), k)
    #encontra o indice do indivíduo com o menor fitness
    posicao = min(selecionados, key=lambda i: valoresDeFitness[i])
    #indivíduo recebe o vencedor na população
    individuo = populacao[posicao]

    #retorna o indivíduo selecionado
    return individuo

## **Crossover**

In [111]:
# ---------------------------------------------------------
# Função de Crossover - Método da Permutação
def crossover(pai1, pai2):
    # gera aleatoriamente o ponto de corte inicial
    pc_inicial = random.sample(range(1, TAMANHO_INDIVIDUO - 3), 1)
    # converte para escalar
    pc_inicial = pc_inicial[0]
    # gera aleatoriamente o ponto de corte final
    pc_final = random.sample(range(pc_inicial + 1, TAMANHO_INDIVIDUO - 2), 1)
    # converte para escalar
    pc_final = pc_final[0]

    # inicializa representação dos filhos com um vetor de -1 (indica que todos os valores estão vazios)
    filho1 = list(np.ones(TAMANHO_INDIVIDUO) * -1)
    filho2 = list(np.ones(TAMANHO_INDIVIDUO) * -1)

    # a partir dos pontos de corte, define a região central de cada pai como a região central de cada filho, respectivamente
    filho1[pc_inicial:pc_final] = pai1[pc_inicial:pc_final]
    filho2[pc_inicial:pc_final] = pai2[pc_inicial:pc_final]

    # inicializa um vetor que indica os valores faltantes nos filhosd
    posFaltantesFilho1 = []
    posFaltantesFilho2 = []

    # percorre cada um dos pais
    for i in range(TAMANHO_INDIVIDUO):
        # verifica se o valor encontrado na posição atual do pai não está presente no filho
        if pai2[i] not in filho1:
            # se a posição correspondente no filho estiver vazia
            if filho1[i] == -1:
                # atribui o valor do pai na posição correspondente do filho
                filho1[i] = pai2[i]
            # senão
            else:
                # salva a posição do pai em que foi encontrado o valor faltante
                posFaltantesFilho1.append(i)

        # repete o processo para o outro pai e o outro filho
        if pai1[i] not in filho2:
            if filho2[i] == -1:
                filho2[i] = pai1[i]
            else:
                posFaltantesFilho2.append(i)

    # percorre cada um dos filhos
    for i in range(TAMANHO_INDIVIDUO):
        # se encontrar um valor vazio no filho
        if filho1[i] == -1:
            # preenche com o primeiro valor que falta que foi encontrado no pai
            filho1[i] = pai2[posFaltantesFilho1.pop(0)]
        # repete processo para o outro filho e o outro pai
        if filho2[i] == -1:
            filho2[i] = pai1[posFaltantesFilho2.pop(0)]

    return filho1, filho2

## **Mutação**

In [112]:
# ---------------------------------------------------------
# Função de Mutação
def mutacao(individuo, mutacao = 0.05):
    # gera um número aleatório
    aleatorio = random.sample(range(100), 1)
    # verifica se o percentual do número aleatório é menor que a taxa de mutação
    if (aleatorio[0]/100 <= mutacao):
        # gera duas posições aletórias para fazer troca
        posicoes = random.sample(range(TAMANHO_INDIVIDUO), 2)
        # enquanto as duas posições geradas forem iguais
        while(posicoes[0] == posicoes[1]):
            # gera novamente duas posições aleatórias
            posicoes = random.sample(range(TAMANHO_INDIVIDUO), 2)
        # troca de valor entre as duas posições (mudança de posição nas colunas)
        individuo[posicoes[0]], individuo[posicoes[1]] = individuo[posicoes[1]], individuo[posicoes[0]]
    # retorna individuo (mutado ou não)
    return individuo


## **Algoritmo Genético**

In [113]:
def algoritmoGenetico (tabuleiro, tamanhoPopulacao = 50, numeroIteracoes = 3000):
    # Gera uma população inicial aleatória
    populacao = [np.zeros(TAMANHO_INDIVIDUO) for i in range(tamanhoPopulacao)]
    # Avalia o fitness da população
    valoresDeFitness = np.zeros(len(populacao))

    # gerando os individuos aleatoriamente
    for i in range(tamanhoPopulacao):
        populacao[i] = random.sample(range(0,8), TAMANHO_INDIVIDUO)
        valoresDeFitness[i] = calcular_conflitos(populacao[i], tabuleiro)

    # inicializa número de iterações
    iteracoes = 0

    # inicializa os parametros de retorno da função
    mediaFitness = []
    id_melhor = min(range(len(populacao)), key=lambda i: valoresDeFitness[i])
    bestIndividuo = populacao[id_melhor]
    bestFitness = valoresDeFitness[id_melhor]

    # média do fitness da população
    mediaFitness.append(float(np.mean(valoresDeFitness)))

    # enquanto iterações é menor que o limite e a melhor fitness é maior que zero
    while (iteracoes < numeroIteracoes and bestFitness > 0):
        novaPopulacao = []
        valoresDeFitnessNovaPopulacao = []

        for k in range(int(tamanhoPopulacao/2)):

            # seleciona dois pais através de torneio
            pai1 = selecaoPorTorneio(populacao, valoresDeFitness, 3)
            pai2 = selecaoPorTorneio(populacao, valoresDeFitness, 3)

            # gera dois filhos através do ruzamento dos pais
            filho1, filho2 = crossover(pai1, pai2)

            # aplica mutação nos filhos gerados
            filho1 = mutacao(filho1, 1/tamanhoPopulacao)
            filho2 = mutacao(filho2, 1/tamanhoPopulacao)

            # adiciona os filhos na nova população
            novaPopulacao.append(filho1)
            novaPopulacao.append(filho2)

            # calcula o fitness da nova população
            valoresDeFitnessNovaPopulacao.append(calcular_conflitos(filho1, tabuleiro))
            valoresDeFitnessNovaPopulacao.append(calcular_conflitos(filho2, tabuleiro))

        # incrementa o número de iterações
        iteracoes += 1

        # atualiza a população
        populacao = novaPopulacao
        valoresDeFitness = valoresDeFitnessNovaPopulacao

        # calcula a media de fitness da população atual
        mediaFitness.append(float(np.mean(valoresDeFitness)))

        id_melhor = min(range(len(populacao)), key=lambda i: valoresDeFitness[i])
        # se novo valor fitness for menor que a melhor (menor) fitness anterior
        if(valoresDeFitness[id_melhor] < bestFitness):
            # atualiza melhor individuo e melhor valor fitness
            bestIndividuo = populacao[id_melhor]
            bestFitness = valoresDeFitness[id_melhor]
    # retorna parametros de avalição
    return (iteracoes, bestIndividuo, bestFitness, mediaFitness, populacao)

# **A\***

# **Pipeline**

In [114]:
def exibe_resultado(estadoFinal, qtdIteracoes, algoritmo):
    if estadoFinal == "FALHA" or estadoFinal is None:
        print("Não foi possível encontrar uma solução para este tabuleiro")
    else:
        print(f"Jogo resolvido em {qtdIteracoes} iterações pelo {algoritmo}!")
        print(f"Posicionamento das rainhas: {estadoFinal}")

        print("\nMatriz Final:")
        tabuleiro_visual = [[0]*8 for _ in range(8)]
        for linha, col in enumerate(estadoFinal):
            tabuleiro_visual[linha][col] = 1
        for linha in tabuleiro_visual:
            print(linha)

In [115]:
def pipeline():
  for id_imagem in range(1, QUANTIDADE_IMAGENS + 1):
    print("########################################################################")
    print(f"####################### MAPEAMENTO TABULEIRO {id_imagem:03d} #######################")
    print("########################################################################\n")
    # função que carrega as imagens (prints) do tabuleiro
    print_tabuleiro = carregar_imagens_tabuleiro(id_imagem)
    print("\n")
    # função que recorta a região de interesse (tabuleiro) do print automaticamente
    tabuleiro_recortado = recortar_imagem_tabuleiro(print_tabuleiro)
    # função que gera a matriz de cores do tabuleiro a partir da imagem do tabuleiro
    matriz_tabuleiro = gerar_matriz_tabuleiro(tabuleiro_recortado)

    # representação dos estados
      # linhas: indice do vetor
      # colunas: valor em cada indice

    print("########################################################################")
    print("Algoritmo Genético (AG)")
    iteracoes, bestIndividuo, bestFitness, mediaFitness, populacao = algoritmoGenetico(matriz_tabuleiro)
    exibe_resultado(bestIndividuo, iteracoes, "AG")

    print("\n\n########################################################################")
    print("Busca em largura (BFS)")
    caminho, estadoFinal, qtdIteracoes = bfs(matriz_tabuleiro)
    exibe_resultado(estadoFinal, qtdIteracoes, "BFS")

    print("\n\n########################################################################")
    print("Busca em profundidade (DFS)")
    caminho, estadoFinal, qtdIteracoes = dfs(matriz_tabuleiro)
    exibe_resultado(estadoFinal, qtdIteracoes, "DFS")

    print("\n\n")

pipeline()

########################################################################
####################### MAPEAMENTO TABULEIRO 001 #######################
########################################################################





TypeError: 'NoneType' object is not subscriptable